# Lab 6: Establishing a Baseline

**Before you start: go to Runtime → Change runtime type and select GPU.**

By now you have a dataset and a working training pipeline. The next step 
is to get your **first result** — a baseline that tells you where you are 
starting from and gives you something concrete to improve on.

A baseline is not a throwaway step. It is one of the most important things 
you will do in your project:

- It confirms your pipeline works end-to-end
- It reveals problems with your data before you invest in complex models
- It gives you a reference point — every subsequent result is either 
  better or worse than the baseline, and you need to explain why
- It appears in your report as the first row of your results table

**This lab covers:**
1. What makes a good baseline
2. Training from scratch vs transfer learning — a direct comparison
3. Choosing the right evaluation metrics for your task
4. Reporting your baseline properly

**Session structure:**
- First 45 minutes: work through this notebook
- Remaining time: run your baseline on your own project data with TA support

**Goal for today:** every group leaves with at least one number to report.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_auc_score,
    average_precision_score
)
import matplotlib.pyplot as plt
import numpy as np
import zipfile, gdown
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load dataset
gdown.download('https://drive.google.com/uc?id=1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj', 'data.zip', quiet=True)
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
data_dir = Path('data')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(data_dir / 'train', transform=train_transform)
val_ds   = datasets.ImageFolder(data_dir / 'val',   transform=val_transform)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes: {CLASS_NAMES}')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

## 1. What makes a good baseline?

A baseline should be:

**Simple** — use the simplest model that could plausibly work. 
If transfer learning is an option (and it almost always is for images), 
a pretrained ResNet-18 with a replaced head is a perfectly good baseline. 
You do not need a novel architecture to establish a reference point.

**Fast to run** — you want results quickly so you can iterate. 
Train for fewer epochs than you think you need at first.

**Reproducible** — set a random seed so your baseline result does not 
change every time you run it.

**Honestly reported** — report validation performance, not training 
performance. A model that memorises the training set is not a useful baseline.

There is one more baseline worth including: **random chance**. 
For a 3-class balanced problem, random chance gives 33.3% accuracy. 
If your model only reaches 40%, that is barely better than guessing — 
and you need to know that early.

In [ ]:
# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Random chance baseline — just for reference
from collections import Counter
label_counts = Counter(train_ds.targets)
total        = sum(label_counts.values())
class_probs  = [label_counts[i] / total for i in range(NUM_CLASSES)]

# Majority class baseline: always predict the most common class
majority_class     = max(label_counts, key=label_counts.get)
majority_acc       = label_counts[majority_class] / total

# Random chance (assuming balanced): 1 / num_classes
random_chance_acc  = 1 / NUM_CLASSES

print('Trivial baselines (lower bounds — your model must beat these):')
print(f'  Random chance:    {random_chance_acc:.3f} ({random_chance_acc*100:.1f}%)')
print(f'  Majority class:   {majority_acc:.3f} ({majority_acc*100:.1f}%)')
print()
print('If your model does not significantly beat these, something is wrong.')

## 2. Training from scratch vs transfer learning

For most image projects, the right question is not *whether* to use 
transfer learning but *how much* of the pretrained model to keep frozen.

We will compare three approaches:
- **From scratch**: random initialisation, train everything
- **Feature extraction**: pretrained backbone frozen, only train the head
- **Full fine-tuning**: pretrained backbone unfrozen, train everything

In [ ]:
def make_model(approach, num_classes):
    """
    Build a model for one of three transfer learning approaches.

    approach: 'scratch' | 'feature_extraction' | 'fine_tuning'
    """
    if approach == 'scratch':
        # Small CNN, no pretrained weights
        return nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((4,4)),
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    # Both transfer approaches start from the same pretrained backbone
    m    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, num_classes)

    if approach == 'feature_extraction':
        # Freeze all layers except the new head
        for name, param in m.named_parameters():
            if 'fc' not in name:
                param.requires_grad = False

    # fine_tuning: everything trainable (default)
    return m


def run_experiment(approach, train_dl, val_dl, epochs=10, lr=1e-3):
    """Train a model and return (model, history, trainable_params)."""
    torch.manual_seed(42)
    model     = make_model(approach, NUM_CLASSES).to(device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    criterion = nn.CrossEntropyLoss()
    # Only pass trainable parameters to the optimiser
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    history   = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

    for epoch in range(1, epochs + 1):
        model.train()
        t_loss = t_corr = t_tot = 0
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * imgs.size(0)
            t_corr += (out.argmax(1) == lbls).sum().item()
            t_tot  += imgs.size(0)

        model.eval()
        v_loss = v_corr = v_tot = 0
        with torch.no_grad():
            for imgs, lbls in val_dl:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out  = model(imgs)
                loss = criterion(out, lbls)
                v_loss += loss.item() * imgs.size(0)
                v_corr += (out.argmax(1) == lbls).sum().item()
                v_tot  += imgs.size(0)

        scheduler.step()
        history['train_acc'].append(t_corr / t_tot)
        history['val_acc'].append(v_corr / v_tot)
        history['train_loss'].append(t_loss / t_tot)
        history['val_loss'].append(v_loss / v_tot)

    best_val = max(history['val_acc'])
    print(f'  [{approach}]  best val acc: {best_val:.3f}  '
          f'trainable params: {trainable:,}')
    return model, history, trainable


print('Running three experiments (this will take a few minutes)...')
results = {}
for approach in ['scratch', 'feature_extraction', 'fine_tuning']:
    print(f'Training: {approach}')
    results[approach] = run_experiment(approach, train_dl, val_dl, epochs=10)

In [ ]:
# Plot all three approaches together
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = {
    'scratch':            'From scratch',
    'feature_extraction': 'Feature extraction',
    'fine_tuning':        'Full fine-tuning',
}
styles = ['--', '-.', '-']
epochs = range(1, 11)

for (approach, (_, history, _)), style in zip(results.items(), styles):
    axes[0].plot(epochs, history['val_acc'],  style, label=labels[approach])
    axes[1].plot(epochs, history['val_loss'], style, label=labels[approach])

axes[0].axhline(random_chance_acc, color='gray', linestyle=':', label='Random chance')
axes[0].set_title('Validation accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Validation loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Three approaches compared on the same dataset', fontsize=13)
plt.tight_layout()
plt.show()

# Summary table
print(f'\n{'Approach':<25} {'Best val acc':>13} {'Trainable params':>18}')
print('-' * 58)
print(f'{'Random chance':<25} {random_chance_acc:>12.3f}')
for approach, (_, history, trainable) in results.items():
    print(f'{labels[approach]:<25} {max(history["val_acc"]):>12.3f} {trainable:>18,}')

## 3. Choosing the right evaluation metrics

Accuracy is not always the right metric. The right choice depends on your task.

### For classification

| Situation | Recommended metrics |
|---|---|
| Balanced classes | Accuracy, per-class accuracy |
| Imbalanced classes | Macro F1, per-class F1, confusion matrix |
| Binary, costs differ | Precision, recall, AUC-ROC |
| Multi-label | Mean Average Precision (mAP) |

### For detection
Mean Average Precision at IoU threshold (mAP@0.5) is the standard metric.
Most detection frameworks compute this for you.

### For segmentation
Mean IoU (Intersection over Union) averaged over classes is the standard.

### For generation
Fréchet Inception Distance (FID) measures image quality and diversity.
Harder to compute — requires generating many samples.

Let's compute the full suite of classification metrics on our best model:

In [ ]:
def evaluate_model(model, val_dl, class_names):
    """
    Compute a comprehensive set of evaluation metrics.
    Returns predictions, labels, and probabilities for further analysis.
    """
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, lbls in val_dl:
            imgs    = imgs.to(device)
            logits  = model(imgs)
            probs   = torch.softmax(logits, dim=1).cpu()
            preds   = logits.argmax(1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(lbls.numpy())
            all_probs.extend(probs.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    n_classes  = len(class_names)

    # --- Accuracy ---
    accuracy = (all_preds == all_labels).mean()

    # --- Classification report (precision, recall, F1 per class) ---
    print('=== Classification Report ===')
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # --- Confusion matrix ---
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(
        confusion_matrix(all_labels, all_preds),
        display_labels=class_names
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion matrix  (accuracy = {accuracy:.3f})')
    plt.tight_layout()
    plt.show()

    # --- Per-class accuracy ---
    print('Per-class accuracy:')
    cm = confusion_matrix(all_labels, all_preds)
    for i, name in enumerate(class_names):
        per_class_acc = cm[i, i] / cm[i].sum()
        print(f'  {name:<20} {per_class_acc:.3f}')

    # --- AUC-ROC (one-vs-rest, for binary or multi-class) ---
    try:
        if n_classes == 2:
            auc = roc_auc_score(all_labels, all_probs[:, 1])
        else:
            auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
        print(f'\nAUC-ROC (one-vs-rest): {auc:.3f}')
    except Exception as e:
        print(f'\nAUC-ROC could not be computed: {e}')

    return all_preds, all_labels, all_probs


# Evaluate the best model (full fine-tuning)
best_model = results['fine_tuning'][0]
preds, labels, probs = evaluate_model(best_model, val_dl, CLASS_NAMES)

## 4. Reporting your baseline

Every result in your report should be reported in a consistent format 
that lets the reader understand exactly what was measured and how.

Here is the minimum information a results table entry needs:
- Model name / approach
- Key hyperparameters (epochs, learning rate)
- Primary metric on the validation set
- Secondary metrics if relevant

Here is a template:

In [ ]:
from sklearn.metrics import f1_score

print('Results table — adapt this format for your project')
print('=' * 75)
print(f'{'Model':<28} {'Acc':>6} {'Macro F1':>9} {'Params':>10} {'Epochs':>7}')
print('-' * 75)

# Trivial baselines
print(f'{'Random chance':<28} {random_chance_acc:>6.3f}  {" ":>9} {" ":>10} {" ":>7}')

# Trained models
for approach, (model, history, trainable) in results.items():
    model.eval()
    ap, al = [], []
    with torch.no_grad():
        for imgs, lbls in val_dl:
            ap.extend(model(imgs.to(device)).argmax(1).cpu().numpy())
            al.extend(lbls.numpy())
    ap, al = np.array(ap), np.array(al)
    acc = (ap == al).mean()
    f1  = f1_score(al, ap, average='macro')
    print(f'{labels[approach]:<28} {acc:>6.3f} {f1:>9.3f} {trainable:>10,} {10:>7}')

print('=' * 75)
print('\nNote: all results on validation set. Epochs=10, lr=1e-3 for all models.')

## 5. Now run your baseline

For the rest of the session, apply what you have learned to your own project.

**Step 1 — Choose your baseline approach.**
For most image projects: start with full fine-tuning of a pretrained ResNet-18.
It is fast, reliable, and gives a strong result to improve on.
If your task is not standard classification, ask a TA.

**Step 2 — Adapt the pipeline from Lab 5.**
Load your dataset, plug in your model, and run for 10 epochs.
Set `torch.manual_seed(42)` so your result is reproducible.

**Step 3 — Choose your metrics.**
Look at the table in section 3 and identify the right metrics for your task.
Run `evaluate_model` or adapt it for your task.

**Step 4 — Record your baseline result.**
Write it down in the format of the results table above.
This is the first row of the results table in your final report.

**Questions to be able to answer before you leave:**
- What is your baseline validation accuracy (or equivalent metric)?
- Does it significantly beat random chance?
- Which class is your model worst at, and why might that be?
- What is the first thing you will try to improve it?

---
**Note on metrics for non-classification tasks:**
If your project involves detection, segmentation, or generation, 
the metrics above do not apply directly. Ask a TA to help you identify 
the right metric and how to compute it in PyTorch.